In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import json, zipfile

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score, confusion_matrix
from baseline import calculate_resilience_cost
SEED = 42
np.random.seed(SEED)

# --- Load data ---
df = pd.read_csv("data/train.csv", index_col=0)
test_df = pd.read_csv("data/test.csv", index_col=0)
cost_matrix = pd.read_csv("data/cost_matrix.csv", index_col=0).values

display(df)

In [ ]:

# --- Encode target and split ---
enc = LabelEncoder().fit(df["alert"])
y = enc.transform(df["alert"])
X = df.drop("alert", axis=1)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)


In [ ]:
# --- Scale features ---
scaler = MinMaxScaler().fit(X_train)
X_train, X_val, X_test = (
    scaler.transform(X_train),
    scaler.transform(X_val),
    scaler.transform(test_df),
)

In [ ]:

# --- Train baseline model ---
model = KNeighborsClassifier(n_neighbors=10)
model.fit(X_train, y_train)
model


In [ ]:


# --- Validate ---
y_val_pred = model.predict(X_val)
f1 = f1_score(y_val, y_val_pred, average="macro")
cm = confusion_matrix(enc.inverse_transform(y_val), enc.inverse_transform(y_val_pred))
rci = calculate_resilience_cost(cm, cost_matrix)

print(f"F1 (macro): {f1:.3f}")
print("Confusion matrix:\n", cm)
print(f"Resilience Cost: {rci:.2f}")